In [1]:
import tensorflow as tf
import autokeras as ak
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datetime import datetime


In [2]:
physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs:", physical_devices)


Num GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
os.listdir()

['sp_titanic.ipynb', 'test.csv', 'train.csv']

In [7]:
data = pd.read_csv(r'..\data\train.csv')
test = pd.read_csv(r'..\data\test.csv')


In [10]:
X = data.drop(columns=['Transported','PassengerId'])  # Remove the target column
y = data['Transported']  # Extract the target column

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [12]:
# Initialize the AutoKeras Structured Data Classifier with column details
clf = ak.StructuredDataClassifier(
    max_trials=3
)

# Fit the model using extracted X and y
clf.fit(X_train, y_train)
model=clf.export_model()
model.save('models/sp_titanic_kaggle', save_format='tf')

Trial 1 Complete [00h 10m 17s]
val_accuracy: 0.8003472089767456

Best val_accuracy So Far: 0.8090277910232544
Total elapsed time: 00h 10m 17s
INFO:tensorflow:Oracle triggered exit
Epoch 1/45
182/182 [==============================] - 2s 6ms/step - loss: 0.5637 - accuracy: 0.7181
Epoch 2/45
182/182 [==============================] - 1s 6ms/step - loss: 0.5079 - accuracy: 0.7653
Epoch 3/45
182/182 [==============================] - 1s 6ms/step - loss: 0.4862 - accuracy: 0.7723
Epoch 4/45
182/182 [==============================] - 1s 6ms/step - loss: 0.4789 - accuracy: 0.7770
Epoch 5/45
182/182 [==============================] - 1s 6ms/step - loss: 0.4736 - accuracy: 0.7807
Epoch 6/45
182/182 [==============================] - 1s 6ms/step - loss: 0.4698 - accuracy: 0.7802
Epoch 7/45
182/182 [==============================] - 1s 6ms/step - loss: 0.4595 - accuracy: 0.7812
Epoch 8/45
182/182 [==============================] - 1s 6ms/step - loss: 0.4632 - accuracy: 0.7867
Epoch 9/45
182/182 [

In [13]:
predicted_y = clf.predict(X_test)
test_loss, test_acc = clf.evaluate(X_test,
                                    y_test,
                                    verbose=0) 
print(test_loss, test_acc)

#predicted_y

90/90 [==============================] - 0s 3ms/step
0.49616000056266785 0.767863392829895


In [14]:
predicted_y = clf.predict(test.drop(columns=['PassengerId']))

134/134 [==============================] - 0s 3ms/step


In [34]:
sub = pd.merge(test[['PassengerId']], pd.DataFrame(predicted_y), left_index=True, right_index=True)#.to_csv('data/ak_submission.csv', index=False)
sub = sub.rename(columns={0:'Transported'})
sub['Transported'] = sub['Transported'].map({0:False, 1:True})
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submission_{timestamp}.csv"
filepath = os.path.join(r"../submissions", filename)  # Cross-platform

# Save DataFrame to CSV
sub.to_csv(filepath, index=False)
